# 05 — Deployment and Consumption

Show all three ways to consume the trained model: CLI, REST API, and Google Sheets automation.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

## Option A — Command-Line Interface (CLI)

The package exposes a CLI via the `health-insurance-cross-sell` entry point (configured in `pyproject.toml`).

```bash
# Train the model
health-insurance-cross-sell train

# Score the test set
health-insurance-cross-sell predict \
  --input data/raw/test.csv \
  --output data/processed/predictions.csv
```

Or using the Makefile shortcuts:

```bash
make train
make predict
```

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "-m", "health_insurance_cross_sell.cli", "--help"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
    env={**__import__("os").environ, "PYTHONPATH": str(PROJECT_ROOT / "src")},
)
print(result.stdout or result.stderr)

## Option B — REST API with FastAPI

Start the server in a terminal:

```bash
pip install -r requirements-api.txt
uvicorn health_insurance_cross_sell.api:app --host 0.0.0.0 --port 8000 --reload
```

The API exposes:
- `GET /` — health check
- `POST /predict` — batch scoring
- `GET /docs` — interactive Swagger UI

### API Request Example

In [ ]:
import json

sample_payload = {
    "records": [
        {
            "id": 1,
            "Gender": "Male",
            "Age": 44,
            "Driving_License": 1,
            "Region_Code": 28.0,
            "Previously_Insured": 0,
            "Vehicle_Age": "> 2 Years",
            "Vehicle_Damage": "Yes",
            "Annual_Premium": 40454.0,
            "Policy_Sales_Channel": 26.0,
            "Vintage": 217
        },
        {
            "id": 2,
            "Gender": "Male",
            "Age": 76,
            "Driving_License": 1,
            "Region_Code": 3.0,
            "Previously_Insured": 0,
            "Vehicle_Age": "1-2 Year",
            "Vehicle_Damage": "No",
            "Annual_Premium": 33536.0,
            "Policy_Sales_Channel": 26.0,
            "Vintage": 183
        }
    ]
}

print("Sample request payload:")
print(json.dumps(sample_payload, indent=2))

In [ ]:
# Uncomment and run ONLY if the API server is running at localhost:8000

# import requests
# response = requests.post(
#     "http://localhost:8000/predict",
#     json=sample_payload,
#     timeout=10,
# )
# print("Status:", response.status_code)
# print(json.dumps(response.json(), indent=2))

### Expected API Response

```json
{
  "prediction": [1, 0],
  "score": [0.847, 0.123]
}
```

Scores range from 0 to 1. Sort the customer list by `score` descending for the priority call list.

## Option C — Google Sheets Automation

### Step 1 — Apps Script (no Python needed)

1. Open your Google Sheet containing customer data.
2. Go to **Extensions → Apps Script**.
3. Paste the contents of `integrations/google_sheets_appscript.gs`.
4. In the Apps Script editor, click **Project Settings (gear icon) → Script Properties**.
5. Add property `CROSS_SELL_API_URL` with the value of your deployed API URL.
6. Click **Save**.
7. Back in the spreadsheet, click **Cross Sell → Score customers**.

The script will:
- Read all rows in the active sheet
- POST them to the API
- Write `prediction` and `cross_sell_score` columns
- Sort rows by score descending
- Highlight rows with score > 0.5 in green
- Show a toast notification when done

### Step 2 — Python automated scoring (alternative)

Use `scripts/score_to_sheets.py` to score a CSV and write results directly to Google Sheets via the Sheets API:

```bash
# Install dependencies
pip install gspread google-auth google-auth-oauthlib

# Set up OAuth2 credentials (see script docstring for details)

# Run
python scripts/score_to_sheets.py \
  --input data/raw/test.csv \
  --sheet-id YOUR_GOOGLE_SHEET_ID
```

In [ ]:
script_path = PROJECT_ROOT / "scripts" / "score_to_sheets.py"
print(script_path.read_text())

## Direct Python Scoring (no API required)

In [ ]:
import pandas as pd
import joblib
from health_insurance_cross_sell.config import load_config
from health_insurance_cross_sell.features import model_matrix, prepare_features

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
model_path = PROJECT_ROOT / "models" / "model.joblib"

if model_path.exists():
    model = joblib.load(model_path)

    test = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "test.csv")
    prepared = prepare_features(test.copy(), config, training=False)
    X, _, _ = model_matrix(prepared, config, training=False)

    scores = model.predict_proba(X)[:, 1]
    result = test.copy()
    result["score"] = scores
    result = result.sort_values("score", ascending=False).reset_index(drop=True)

    output_path = PROJECT_ROOT / "data" / "processed" / "predictions.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)

    print(f"Scored {len(result):,} customers. Top 5 by score:")
    print(result[["id", "score"]].head())
    print(f"\nSaved to: {output_path}")
else:
    print(f"Model not found at {model_path}. Run notebook 04 first (or `make train`).")